In [1]:
import oceanbench

oceanbench.__version__

'0.5.1'

### Open challenger datasets

> Insert here the code that opens the challenger dataset as `challenger_dataset: xarray.Dataset`

In [2]:
# glowcascade_final over the full official start set, cut to NINE lead days.
#
# 52 Wednesday challenger folders of 2024, 20240103 through 20241225, each
# initialised from the as-issued GLO12 nowcast of the Tuesday before it and
# forced by the IFS forecast issued that same Tuesday.
#
# WHY NINE. The IFS forecast package carries lead_day_index 0..9, that is the
# forcing of forecast days 1..10 minus its last day, so the tenth forecast day
# is driven by PERSISTED lead 9 forcing rather than by a forecast. Julien's
# decision of 2026-09-10: the entry scores the nine days that are forced by a
# real IFS forecast and stops there. Nothing is recomputed and no forecast
# zarr is touched: the store still holds ten days per start, this module drops
# the last time step on the way in.
#
# This copy is scored under oceanbench origin/main 7e5ec87 (pending 0.6.0).
import datetime
import pathlib

import xarray

_ROOT = pathlib.Path("/mnt/data/glonet2/ifs21/forecasts/glowcascade_v5_ring")  # remasked in place 2026-09-21
_PATHS = sorted(_ROOT.glob("2024*.zarr"))
_FIRST_DAYS = [datetime.datetime.strptime(p.stem, "%Y%m%d") for p in _PATHS]
_LEAD_DAYS = 9


def _prepared(dataset: xarray.Dataset) -> xarray.Dataset:
    dataset = dataset.isel(time=slice(0, _LEAD_DAYS))
    lead_count = dataset.sizes["time"]
    return dataset.rename({"time": "lead_day_index"}).assign_coords({"lead_day_index": range(lead_count)})


challenger_dataset: xarray.Dataset = xarray.open_mfdataset(
    [str(p) for p in _PATHS],
    engine="zarr",
    preprocess=_prepared,
    combine="nested",
    concat_dim="first_day_datetime",
    parallel=False,
).assign_coords({"first_day_datetime": _FIRST_DAYS})


### Evaluation configuration

In [3]:
region = 'global'

### Evaluation of challenger dataset using OceanBench

#### Root Mean Square Deviation (RMSD) of variables compared to GLORYS reanalysis

In [4]:
oceanbench.metrics.rmsd_of_variables_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.066689,0.067094,0.067126,0.067202,0.067560,0.068174,0.069080,0.070309,0.071286
Temperature (°C) [sea_water_potential_temperature]{surface},0.516503,0.517104,0.517520,0.519694,0.524064,0.531667,0.542523,0.554619,0.564399
Salinity (PSU) [sea_water_salinity]{surface},0.571705,0.567749,0.563775,0.560293,0.556730,0.553744,0.551338,0.549127,0.546310
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.118529,0.119225,0.120081,0.121291,0.122870,0.125125,0.127970,0.131203,0.133770
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.119565,0.120312,0.121291,0.122660,0.124715,0.127334,0.130412,0.133647,0.136178
Temperature (°C) [sea_water_potential_temperature]{50m},0.864365,0.864311,0.863186,0.862006,0.862435,0.864149,0.867353,0.871824,0.873896
Salinity (PSU) [sea_water_salinity]{50m},0.242355,0.242235,0.241897,0.241544,0.241356,0.241358,0.241510,0.241706,0.241600
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.112359,0.112924,0.113310,0.113803,0.114536,0.115678,0.117244,0.119069,0.120245
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.113317,0.113524,0.113754,0.114152,0.114764,0.115741,0.117231,0.118929,0.120020
Temperature (°C) [sea_water_potential_temperature]{100m},1.059240,1.059990,1.059745,1.059851,1.060854,1.063641,1.068512,1.073212,1.074875


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLORYS reanalysis

In [5]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},40.896352,41.263743,41.565341,41.824217,42.13967,42.487768,42.889357,43.29638,43.498391


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLORYS reanalysis

In [6]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.116794,0.117416,0.117423,0.117797,0.118502,0.119268,0.121129,0.122879,0.124004
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.122738,0.123018,0.123128,0.122835,0.123478,0.123993,0.126086,0.128280,0.129862


#### Root Mean Square Deviation (RMSD) of variables compared to observations

In [7]:
oceanbench.metrics.rmsd_of_variables_compared_to_observations(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9,Observations
Temperature (°C) [sea_water_potential_temperature]{surface},0.735609,0.759574,0.729410,0.753261,0.775686,0.816247,0.814749,0.830133,0.854162,150475
Temperature (°C) [sea_water_potential_temperature]{0-5m},0.725589,0.738767,0.758897,0.784058,0.784839,0.795780,0.810671,0.808593,0.803414,80886
Temperature (°C) [sea_water_potential_temperature]{5-100m},0.861967,0.872154,0.860224,0.889218,0.880276,0.882314,0.904488,0.934414,0.930380,1341567
Temperature (°C) [sea_water_potential_temperature]{100-300m},0.781372,0.803271,0.793799,0.796249,0.813227,0.811210,0.837014,0.833985,0.864388,2132232
Temperature (°C) [sea_water_potential_temperature]{300-600m},0.519676,0.532600,0.529041,0.528457,0.540218,0.557104,0.557255,0.567978,0.585163,2646322
Salinity (PSU) [sea_water_salinity]{0-5m},0.250921,0.277759,0.269240,0.304332,0.274255,0.279050,0.269200,0.263339,0.292052,69024
Salinity (PSU) [sea_water_salinity]{5-100m},0.195103,0.200343,0.204768,0.204997,0.205372,0.205773,0.208021,0.206867,0.210254,1143321
Salinity (PSU) [sea_water_salinity]{100-300m},0.126606,0.128694,0.129276,0.128808,0.132181,0.130998,0.137162,0.133156,0.136329,1816368
Salinity (PSU) [sea_water_salinity]{300-600m},0.080287,0.080704,0.080086,0.080347,0.081442,0.083150,0.082983,0.086157,0.087259,2245406
Sea level anomaly (m) [sea_surface_height_above_geoid]{surface},0.048036,0.049041,0.049847,0.051272,0.052529,0.054256,0.055876,0.057610,0.059715,15465310


#### Deviation of Lagrangian trajectories compared to GLORYS reanalysis

In [8]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glorys_reanalysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},9.967599,19.256474,28.165468,36.798054,45.222412,53.469769,61.558464


#### Root Mean Square Deviation (RMSD) of variables compared to GLO12 analysis

In [9]:
oceanbench.metrics.rmsd_of_variables_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Sea surface height (m) [sea_surface_height_above_geoid]{surface},0.013604,0.016567,0.019047,0.022206,0.026046,0.030062,0.034310,0.038610,0.041678
Temperature (°C) [sea_water_potential_temperature]{surface},0.171970,0.198739,0.226761,0.256034,0.287123,0.319801,0.355023,0.389310,0.415167
Salinity (PSU) [sea_water_salinity]{surface},0.127489,0.146376,0.162070,0.176157,0.189328,0.204320,0.219445,0.233393,0.244647
Meridional current (m/s) [northward_sea_water_velocity]{surface},0.048676,0.054977,0.061613,0.069149,0.077284,0.085630,0.094509,0.102878,0.109084
Zonal current (m/s) [eastward_sea_water_velocity]{surface},0.050300,0.057022,0.063935,0.071383,0.079695,0.088388,0.097160,0.105605,0.111940
Temperature (°C) [sea_water_potential_temperature]{50m},0.303664,0.329085,0.354906,0.383459,0.415642,0.451741,0.490091,0.525269,0.546997
Salinity (PSU) [sea_water_salinity]{50m},0.062802,0.067467,0.072822,0.078765,0.085404,0.092635,0.100186,0.107250,0.112012
Meridional current (m/s) [northward_sea_water_velocity]{50m},0.043708,0.047893,0.052846,0.058353,0.064677,0.071545,0.078931,0.085915,0.090748
Zonal current (m/s) [eastward_sea_water_velocity]{50m},0.045750,0.049863,0.054669,0.060146,0.066270,0.072967,0.080093,0.087041,0.092021
Temperature (°C) [sea_water_potential_temperature]{100m},0.272771,0.304829,0.338796,0.377599,0.419707,0.464820,0.511845,0.554676,0.581283


#### Root Mean Square Deviation (RMSD) of Mixed Layer Depth (MLD) compared to GLO12 analysis

In [10]:
oceanbench.metrics.rmsd_of_mixed_layer_depth_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Mixed layer depth (m) [ocean_mixed_layer_thickness]{surface},30.805253,31.819678,32.628063,33.4608,34.324535,35.219183,36.138222,36.932078,37.576067


#### Root Mean Square Deviation (RMSD) of geostrophic currents compared to GLO12 analysis

In [11]:
oceanbench.metrics.rmsd_of_geostrophic_currents_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 1,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8,Lead day 9
Meridional geostrophic current (m/s) [geostrophic_northward_sea_water_velocity]{surface},0.043764,0.050615,0.057538,0.064696,0.071740,0.079016,0.086457,0.093350,0.098029
Zonal geostrophic current (m/s) [geostrophic_eastward_sea_water_velocity]{surface},0.045598,0.053613,0.061581,0.069269,0.076772,0.084647,0.092393,0.099386,0.103963


#### Deviation of Lagrangian trajectories compared to GLO12 analysis

In [12]:
oceanbench.metrics.deviation_of_lagrangian_trajectories_compared_to_glo12_analysis(
    challenger_dataset,
    region=region,
)

,Lead day 2,Lead day 3,Lead day 4,Lead day 5,Lead day 6,Lead day 7,Lead day 8
Lagrangian trajectory deviation (km) []{surface},3.829596,7.583349,11.556619,15.896077,20.672014,25.91276,31.602322
